# Analisi della Dinamica di Convergenza nel Tempo
## Studio e Sperimentazione AE-XAD su Dataset BUSI

### 📌 Descrizione dell'Esperimento
Questo notebook costituisce l'**orchestratore per lo studio di convergenza temporale**. Monitora e confronta l'andamento della metrica di localizzazione (**pixel-AUC media ± SEM sui 4 seed deterministici: 101, 359, 461, 491**) per tutte le quattro architetture testate nell'intervallo temporale compreso tra **100 e 250 epoche** (valutate a step di 50 epoche: 100, 150, 200, 250).

### 🎯 Motivazioni e Conclusioni della Tesi
1. **Comportamento della Baseline e del Bottleneck Denso:** Entrambe mostrano un degrado progressivo delle prestazioni oltre le 100–150 epoche a causa del mancato contenimento della capacità espressiva.
2. **Stabilità delle Convoluzionali Compresse (Arch. 2a e 2b):** Mantengono una stabilità elevata e costante nell'intero intervallo 100–200 epoche.
3. **Picco a 200 Epoche di Arch. 2b (32×28):** Raggiunge il record assoluto di localizzazione a **200 epoche ($0{,}8722$)**, motivando formalmente la selezione di 200 epoche come punto di arresto ottimale per tutti gli esperimenti comparativi della tesi.


In [ ]:
import sys
from pathlib import Path

# Project root resolution (works both from repo root and experiments/ folder)
PROJECT_ROOT = Path("..").resolve() if (Path("..") / "repo").exists() else Path(".").resolve()
REPO_DIR = PROJECT_ROOT / "repo" / "AE-XAD-main"

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"AE-XAD repo: {REPO_DIR}")


In [ ]:
ARCH 1: baseline ===
run_arch("baseline", "run_224_noLayer_folder", "conv_baseline")

In [ ]:
ARCH 2a: 64x14 ===
run_arch("64x14", "run_224_newLayer_folder_64x14", "conv_64x14")

In [ ]:
ARCH 2b: 32x28 ===
run_arch("32x28", "run_224_newLayer_folder_32x28", "conv_32x28")

In [ ]:
ARCH 3: dense ===
run_arch("dense", "run_224_newLayer_folder_dense", "conv_dense")

In [ ]:
import json
from collections import defaultdict
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

OUT_PNG = CONV_DIR / "report_figures" / "convergenza_inter_arch.png"
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)

LABELS = {
    "baseline": "baseline 64x28",
    "64x14":    "64x14",
    "32x28":    "32x28",
    "dense":    "denso",
}
data = json.loads(MASTER_JSON.read_text())
plt.figure(figsize=(8, 5))
for key, legend in LABELS.items():
    seed_dict = data.get(key, {})
    if not seed_dict:
        print(f"  nessun dato per {key}"); continue
    by_epoch = defaultdict(list)
    for runs in seed_dict.values():
        for pt in runs:
            epoca = int(pt["epoch"])
            if 100 <= epoca <= 250:
                by_epoch[epoca].append(float(pt["auc"]))
                
    if not by_epoch:
        continue
        
    epochs = sorted(by_epoch)
    mean = [np.mean(by_epoch[e]) for e in epochs]
    plt.plot(epochs, mean, marker="o", linewidth=2, label=legend)

plt.xlabel("Epoche"); plt.ylabel("pixel-AUC (media sui seed)")
plt.title("Convergenza per architettura")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig(OUT_PNG, dpi=150)
print("Grafico salvato in", OUT_PNG)
plt.show()

In [ ]:
PLOT: convergenza per architettura — media sui seed ± SEM ===
import json
from collections import defaultdict
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

OUT_PNG = CONV_DIR / "report_figures" / "convergenza_inter_arch_sem.png"
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)

LABELS = {
    "baseline": "baseline 64x28",
    "64x14":    "64x14",
    "32x28":    "32x28",
    "dense":    "denso",
}

data = json.loads(MASTER_JSON.read_text())
plt.figure(figsize=(8, 5))

for key, legend in LABELS.items():
    seed_dict = data.get(key, {})
    if not seed_dict:
        print(f"  nessun dato per {key}"); continue

    by_epoch = defaultdict(list)
    for runs in seed_dict.values():
        for pt in runs:
            epoca = int(pt["epoch"])
            if 100 <= epoca <= 250:
                by_epoch[epoca].append(float(pt["auc"]))
                
    if not by_epoch:
        continue

    epochs = sorted(by_epoch)
    mean = np.array([np.mean(by_epoch[e]) for e in epochs])
    std  = np.array([np.std(by_epoch[e], ddof=1) for e in epochs])   # σ tra seed (campionaria)
    n    = np.array([len(by_epoch[e]) for e in epochs])
    band = std / np.sqrt(n)            # SEM = σ/√n  (deviazione standard della media)
    # band = std                      # <-- decommenta per banda a σ piena invece del SEM

    line, = plt.plot(epochs, mean, marker="o", linewidth=2, label=legend)
    plt.fill_between(epochs, mean - band, mean + band,
                     color=line.get_color(), alpha=0.18, linewidth=0)

plt.xlabel("Epoche")
plt.ylabel("pixel-AUC (media sui seed)")
plt.title("Convergenza per architettura (banda = SEM, n=4 seed)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_PNG, dpi=150)
print("Grafico salvato in", OUT_PNG)
plt.show()

In [ ]:
PLOT: dense vs 64x14 — stabilità inter-seed  ===
import json
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

OUT_PNG = CONV_DIR / "report_figures" / "dense_vs_64x14_stabilita.png"
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)

data = json.loads(MASTER_JSON.read_text())

def series(key):
    sd = data[key]
    by_epoch = defaultdict(list)
    per_seed = {}
    for seed, runs in sd.items():
        ex, ey = [], []
        for pt in runs:
            e = int(pt["epoch"]); a = float(pt["auc"])
            by_epoch[e].append(a); ex.append(e); ey.append(a)
        per_seed[seed] = (ex, ey)
    epochs = sorted(by_epoch)
    mean = np.array([np.mean(by_epoch[e]) for e in epochs])
    std  = np.array([np.std(by_epoch[e], ddof=1) for e in epochs])
    return epochs, mean, std, per_seed

panels = [("64x14", "64x14"), ("dense", "dense")]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)

for ax, (key, title) in zip(axes, panels):
    epochs, mean, std, per_seed = series(key)
    for seed, (ex, ey) in per_seed.items():          # singoli seed dietro
        order = np.argsort(ex)
        ax.plot(np.array(ex)[order], np.array(ey)[order],
                color="gray", alpha=0.35, linewidth=0.9)
    ax.fill_between(epochs, mean - std, mean + std,   # banda ±σ tra seed
                    alpha=0.18, color="tab:blue", linewidth=0)
    ax.plot(epochs, mean, marker="o", linewidth=2.4,  # media in evidenza
            color="tab:blue", label="media 4 seed")
    ax.set_title(f"{title}\nσ a 50/100: {std[0]:.3f}/{std[1]:.3f}")
    ax.set_xlabel("Epoche")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("pixel-AUC")
axes[0].legend(loc="lower left")
plt.suptitle("Stabilità inter-seed: linee grigie = singoli seed, banda = ±σ", y=1.02)
plt.tight_layout()
plt.savefig(OUT_PNG, dpi=150, bbox_inches="tight")
print("Grafico salvato in", OUT_PNG)
plt.show()

In [ ]:
import json, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SEED_DIR = PROJECT_ROOT
TARGET   = "benign (315).png"

gt  = np.load(SEED_DIR / "gtmaps.npy")
taf = json.loads((SEED_DIR / "test_anom_files.json").read_text())

idx = next((i for i,e in enumerate(taf) if e["fname"] == TARGET), None)
print("shape gtmaps:", gt.shape, "| indice target:", idx)

if idx is not None:
    m = gt[idx].squeeze()
    n = int((m > 0).sum())
    print(f"pixel anomali nella maschera (224x224): {n}")
    print(f"atteso ~ {8256 * (224*224) / (550*430):.0f} px se fusa")  # stima grezza riscalata
    print("-> se vedi DUE blob e n e' molto > 676, la fusione e' nel gtmaps. ")
    plt.imshow(m, cmap="gray"); plt.title(f"{TARGET}  ({n}px)"); plt.show()
else:
    print("Target non in test per questo seed — scegli un altro file multi-maschera.")

In [ ]:
import json
data = json.load(open(str(PROJECT_ROOT / "convergence_results.json")))
for s,pts in data["32x28"].items():
    print(s, {p["epoch"]: round(p["auc"],3) for p in pts})

In [ ]:
import json
import pandas as pd

with open(MASTER_JSON) as f:
    data = json.load(f)

EPOCHS_TO_SHOW = [50, 100, 150, 200]
ARCH_ORDER = ["baseline", "64x14", "32x28", "dense"]

rows = []
for ep in EPOCHS_TO_SHOW:
    row = {"epoca": ep}
    for arch in ARCH_ORDER:
        vals = [next(p["auc"] for p in pts if p["epoch"] == ep)
                 for pts in data[arch].values()]
        row[arch] = sum(vals) / len(vals)
    rows.append(row)

df_medie = pd.DataFrame(rows).set_index("epoca").round(4)
df_medie

In [ ]:
import json
import statistics

CONVERGENCE_JSON = str(PROJECT_ROOT / "convergence_results.json")

with open(CONVERGENCE_JSON) as f:
    data = json.load(f)

ARCH_ORDER = ["baseline", "64x14", "32x28", "dense"]

for arch in ARCH_ORDER:
    vals = [next(p["auc"] for p in pts if p["epoch"] == 200) for pts in data[arch].values()]
    mu = statistics.mean(vals)
    sd = statistics.pstdev(vals)
    print(f"{arch:10s}: mu={mu:.4f}  sigma={sd:.4f}")

In [ ]:
import json
import numpy as np
from pathlib import Path

RUNS_BASE_32x28_AUG = PROJECT_ROOT
SEEDS = [101, 359, 461, 491]

benign_means, malignant_means = [], []

for seed in SEEDS:
    with open(RUNS_BASE_32x28_AUG / f"seed{seed}" / "metrics_per_class.json") as f:
        m = json.load(f)
    b, mal = m["per_class"]["benign"]["auc"], m["per_class"]["malignant"]["auc"]
    benign_means.append(b)
    malignant_means.append(mal)
    print(f"seed {seed}: benign={b:.4f}  malignant={mal:.4f}")

print()
print(f"benign:    mu={np.mean(benign_means):.4f}  sigma={np.std(benign_means):.4f}")
print(f"malignant: mu={np.mean(malignant_means):.4f}  sigma={np.std(malignant_means):.4f}")

In [ ]:
import json
import numpy as np
from pathlib import Path

RUNS_BASE_32x28_NOAUG = PROJECT_ROOT
SEEDS = [101, 359, 461, 491]

total_means, benign_means, malignant_means = [], [], []

for seed in SEEDS:
    with open(RUNS_BASE_32x28_NOAUG / f"seed{seed}" / "metrics_per_class.json") as f:
        m = json.load(f)
    tot = m["total"]["auc"]
    b   = m["per_class"]["benign"]["auc"]
    mal = m["per_class"]["malignant"]["auc"]
    total_means.append(tot)
    benign_means.append(b)
    malignant_means.append(mal)
    print(f"seed {seed}: totale={tot:.4f}  benign={b:.4f}  malignant={mal:.4f}")

print()
print(f"TOTALE:    mu={np.mean(total_means):.4f}  sigma={np.std(total_means):.4f}")
print(f"benign:    mu={np.mean(benign_means):.4f}  sigma={np.std(benign_means):.4f}")
print(f"malignant: mu={np.mean(malignant_means):.4f}  sigma={np.std(malignant_means):.4f}")